In [34]:
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch



train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((32, 32)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


base_dataset = ImageFolder(root=r'C:\github\dataset\archive\train')

train_amount = int(0.8 * len(base_dataset))
val_amount = len(base_dataset) - train_amount

generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(
    range(len(base_dataset)),
    [train_amount, val_amount],
    generator=generator
)

train_folder = ImageFolder(
    root=r'C:\github\dataset\archive\train',
    transform=train_transforms
)

val_folder = ImageFolder(
    root=r'C:\github\dataset\archive\train',
    transform=val_transforms
)

train_set = torch.utils.data.Subset(train_folder, train_indices.indices)
val_set = torch.utils.data.Subset(val_folder, val_indices.indices)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [35]:
import torch
import torch.nn as nn
import numpy as np
import torchvision

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)


In [37]:
num_classes = 2

model.conv1 = nn.Conv2d(in_channels=3, out_channels=64,kernel_size=3,stride=1,padding=1,bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.fc.in_features,num_classes)
)
model = model.to(device)


In [38]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 999

In [39]:
print(device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

cuda
True
NVIDIA GeForce RTX 4080 Laptop GPU


In [40]:
early_stopping_hyperparam = 10
early_stopping_epoch = 0
best_val_loss = None


for e in range(epochs):
    print(f'Epochs {e + 1}:')
    model.train()
    print('TRAINING...')

    total_train_loss = 0
    total_train_acc =0
    train_correct = 0
    num_batches = 0
    for batch in train_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        predictions = model(images)
        loss = loss_fn(predictions, labels)
        predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
        correct = (predicted_classes == labels).sum().item()
        
        num_batches += 1
        total_train_acc += correct
        total_train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Training Loss: {total_train_loss/num_batches}')
    print(f'Training Acc: {(total_train_acc/(num_batches *32)) * 100:.2f}%')

    model.eval()
    print('VALIDATING...')
    total_val_loss = 0
    total_val_acc =0
    val_correct = 0
    num_batches = 0
    with torch.no_grad():
        for val_batch in val_loader:
            images, labels = val_batch
            images, labels = images.to(device), labels.to(device)
            predictions = model(images)

            predicted_classes = torch.argmax(predictions, dim=1)  # FIX: add dim=1
            correct = (predicted_classes == labels).sum().item()

            num_batches += 1
            total_val_acc += correct
            total_val_loss += loss_fn(predictions, labels).item()


    print(f'Val Loss: {total_val_loss/num_batches}')
    print(f'Val Acc: {(total_val_acc/(num_batches * 32)) * 100:.2f}%')
    if not best_val_loss or total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        early_stopping_epoch = 0
        print("Saving model")
        torch.save(model.state_dict(), 'model/best_model.pth')
    else:
        early_stopping_epoch += 1
        if early_stopping_epoch >= early_stopping_hyperparam:
            print('Early stopping triggered. Finish Traning.')
            break




Epochs 0:
TRAINING...
Training Loss: 0.2513727403640747
Training Acc: 89.85%
VALIDATING...
Val Loss: 0.15706984658241271
Val Acc: 94.03%
Saving model
Epochs 1:
TRAINING...
Training Loss: 0.18334554652124643
Training Acc: 93.00%
VALIDATING...
Val Loss: 0.19868500816822052
Val Acc: 91.48%
Epochs 2:
TRAINING...
Training Loss: 0.15663658285960555
Training Acc: 94.02%
VALIDATING...
Val Loss: 0.12176965072229505
Val Acc: 95.28%
Saving model
Epochs 3:
TRAINING...
Training Loss: 0.1420213167771697
Training Acc: 94.65%
VALIDATING...
Val Loss: 0.17211694855690002
Val Acc: 93.61%
Epochs 4:
TRAINING...
Training Loss: 0.12955956450141967
Training Acc: 95.09%
VALIDATING...
Val Loss: 0.10900169080272316
Val Acc: 95.94%
Saving model
Epochs 5:
TRAINING...
Training Loss: 0.11534453676138073
Training Acc: 95.71%
VALIDATING...
Val Loss: 0.117288652398251
Val Acc: 95.69%
Epochs 6:
TRAINING...
Training Loss: 0.10760813748426735
Training Acc: 95.96%
VALIDATING...
Val Loss: 0.10003984967377037
Val Acc: 96.40%